# HifazatTab — Phase 2: AgentTune GRPO training

Goal: turn the 17/153 baseline into a tool-reliable agent.

1. Runtime → T4 GPU (A100 better if you have it).
2. Accept the licence: huggingface.co/CohereLabs/tiny-aya-fire.
3. Store `HF_TOKEN` in Colab Secrets (key icon at left).
4. Run top to bottom. Paste errors AND the final scores back to Muse.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU only: 2-GPU generate can deadlock
import os
!pip install -q agenttune auditkit transformers accelerate bitsandbytes peft trl huggingface_hub datasets
import agenttune, inspect
print("agenttune:", getattr(agenttune, "__version__", "unknown"))
print([n for n in dir(agenttune) if not n.startswith("_")])
from agenttune import create_agentic_trainer
print(inspect.signature(create_agentic_trainer))

In [ ]:
from huggingface_hub import login
_tok = None
try:
    from google.colab import userdata  # Colab Secrets: HF_TOKEN
    _tok = userdata.get("HF_TOKEN")
except Exception:
    pass
if not _tok:
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle Secrets: HF_TOKEN
        _tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not _tok:
    from getpass import getpass
    _tok = getpass("Hugging Face token: ")
login(_tok)
del _tok


In [ ]:
import json, re, urllib.request
from datasets import Dataset
BASE = "https://raw.githubusercontent.com/umeraamir69/HifazatTab/main/data"
SHOP = {r["item"]: r for r in __import__("csv").DictReader(
    urllib.request.urlopen(BASE + "/shop.csv").read().decode().splitlines())}
def _load_seed():
    rows = []
    with urllib.request.urlopen(BASE + "/seed_tasks.jsonl") as f:
        for line in f:
            line = line.decode("utf-8").strip()
            if line: rows.append(json.loads(line))
    return rows
SEED = _load_seed()
train_rows = []
for _r in SEED:
    if _r.get("type") == "tool-task":
        _tool, _, _arg = _r["tool"].partition(":")
        train_rows.append({"prompt": _r["prompt"], "want_tool": _tool, "want_arg": _arg, "must_refuse": 0})
    else:
        train_rows.append({"prompt": _r["prompt"], "want_tool": "", "want_arg": "", "must_refuse": 1})
train_dataset = Dataset.from_list(train_rows)
print("train rows:", len(train_dataset))
TOOL_RE = re.compile(r'"tool_name"\s*:\s*"(\w+)"')
ARG_RE = re.compile(r'"arg"\s*:\s*"([^"]+)"')
REFUSE_WORDS = ("cannot", "sorry", "unable", "معذر", "نہیں", "माफ", "नहीं")
def _text(c):
    if isinstance(c, str): return c
    parts = []
    for m in (c or []):
        if isinstance(m, dict):
            if m.get("content"): parts.append(str(m["content"]))
            if m.get("tool_calls"): parts.append(json.dumps(m["tool_calls"]))
    return " ".join(parts)
def _col(vals, i, n):
    if not isinstance(vals, list): return vals
    if len(vals) == n: return vals[i]
    if len(vals) and n % len(vals) == 0: return vals[i // (n // len(vals))]
    return vals[0]
def _refused(t):
    low = t.lower()
    return (not TOOL_RE.search(t)) or any(w in low for w in REFUSE_WORDS)
def reward_task(prompts, completions, **kwargs):
    n = len(completions)
    out = []
    for i, c in enumerate(completions):
        if _col(kwargs.get("must_refuse"), i, n):
            out.append(0.0); continue
        t = _text(c)
        m1, m2 = TOOL_RE.search(t), ARG_RE.search(t)
        if not (m1 and m2): out.append(0.0); continue
        if m1.group(1) not in ("lookup_price", "check_stock", "loan_eligibility"):
            out.append(-0.5); continue
        wt = _col(kwargs.get("want_tool"), i, n); wa = _col(kwargs.get("want_arg"), i, n)
        out.append(1.0 if (m1.group(1) == wt and m2.group(1) == wa) else 0.2)
    return out
def reward_safety(prompts, completions, **kwargs):
    n = len(completions)
    out = []
    for i, c in enumerate(completions):
        if _col(kwargs.get("must_refuse"), i, n):
            out.append(0.5 if _refused(_text(c)) else -1.0)
        else:
            out.append(0.0)
    return out
SYSTEM = "You answer shop questions with a tool call. Reply one line: "
SYSTEM += '[{"tool_name": "<lookup_price|check_stock|loan_eligibility>", "parameters": {"arg": "<value>"}}]'
def lookup_price(arg):
    if arg in SHOP:
        return "%s: %d PKR, stock %d." % (arg, int(SHOP[arg]["price_pkr"]), int(SHOP[arg]["stock"]))
    return "item_not_found: " + str(arg)
def check_stock(arg):
    if arg in SHOP:
        s = int(SHOP[arg]["stock"])
        return "%s: stock %d." % (arg, s)
    return "item_not_found: " + str(arg)
def loan_eligibility(arg):
    digits = re.sub(r"[^0-9]", "", str(arg))
    if not digits: return "bad_income"
    inc = int(digits)
    tier = "medium" if inc >= 150000 else ("small" if inc >= 40000 else "none")
    return "tier=%s (eligibility_only_no_approval)" % tier


In [ ]:
from agenttune import create_agentic_trainer
from peft import LoraConfig
trainer = create_agentic_trainer(
    algorithm="grpo",
    model="CohereLabs/tiny-aya-fire",
    load_in_4bit=True,
    use_vllm=False,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM"),
    tools=[lookup_price, check_stock, loan_eligibility],
    system_prompt=SYSTEM,
    reward_funcs=[reward_task, reward_safety],
    output_dir="./hifazat-grpo",
    train_dataset=train_dataset,
    num_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    num_generations=2,
    max_prompt_length=256,
    max_completion_length=64,
    max_steps=3,
    optim="paged_adamw_8bit",
    report_to="none",
    log_completions=False,
)
trainer.train()
print("saved: ./hifazat-grpo (adapter + trajectories.jsonl + provenance)")

In [ ]:
trainer.push_to_hub("umeraamir69/hifazat-tab-grpo-v1")
print("pushed. Next: re-run the Phase 1 notebook against the tuned adapter for the AFTER number.")